# 04. Conditional buy model: P(TP) + risk | no TP

TP와 downside 관계를 반대로 학습한 전체표본 multi-task loss를 수정한다.

- `P(TP)`는 전체 표본의 weighted BCE
- `E(timeout | no TP)`와 `Q20(downside | no TP)`는 TP 실패 표본에서만 학습
- OOF validation 날짜를 5일로 확장해 다음 매도 모델의 진입 표본을 만든다.
- 마지막 거래일은 최종 Test로 격리한다.

In [1]:
from pathlib import Path
import gc
import json
import math
import time

import nbformat
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import average_precision_score, brier_score_loss, mean_absolute_error, roc_auc_score
from torch import nn
from torch.utils.data import DataLoader


def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'AGENT.md').is_file() and (candidate / 'README.md').is_file():
            return candidate
    raise FileNotFoundError('프로젝트 루트를 찾지 못했습니다.')


PROJECT_ROOT = find_project_root()
source_notebook = nbformat.read(PROJECT_ROOT / 'notebooks/03_multitask_tp_timeout_downside.ipynb', as_version=4)
for cell_index, marker in [(1, 'future_downside'), (3, 'MultiTaskMPTSNet')]:
    source = source_notebook.cells[cell_index].source
    assert marker in source, (cell_index, marker)
    exec(compile(source, f'03_base_cell_{cell_index}', 'exec'), globals())

MODEL_VERSION = 'conditional_buy_mptsnet_v1'
OOF_VALID_SESSIONS = sessions[3:-1]
MAX_EPOCHS = 25
PATIENCE = 5
MIN_OOF_TRADES = 50
RESULT_ROOT = (PROJECT_ROOT / '../../results/modeling').resolve()
MODEL_ROOT = (PROJECT_ROOT / '../../model').resolve()
print('conditional OOF sessions:', OOF_VALID_SESSIONS)
print('final Test:', TEST_SESSION)

device: cuda
X: (72181, 30, 38) | positive: 3.94%
OOF validation: ['session_2026-07-14', 'session_2026-07-15', 'session_2026-07-16']
final Test: session_2026-07-17
model version: multitask_mptsnet_tp_timeout_downside_v1
timeout return percentiles: [-0.09101166 -0.00600253  0.086385  ]
downside percentiles: [-0.11480958 -0.01460211 -0.00099062]
conditional OOF sessions: ['session_2026-07-10', 'session_2026-07-13', 'session_2026-07-14', 'session_2026-07-15', 'session_2026-07-16']
final Test: session_2026-07-17


## Conditional loss와 expanding OOF

회귀 head는 `y=0` batch row만 사용한다. utility는 `P(TP)×3% + (1-P(TP))×E(timeout|no TP)`에서 조건부 downside penalty를 차감한다.

In [2]:
@torch.no_grad()
def predict_conditional(model, x_scaled, indices):
    loader = DataLoader(MultiTaskDataset(x_scaled, indices), batch_size=BATCH_SIZE, shuffle=False)
    model.eval()
    result = {'index': [], 'tp_probability': [], 'predicted_timeout_no_tp': [], 'predicted_downside_no_tp': []}
    for batch_x, _, _, _, batch_indices in loader:
        output = model(batch_x.to(DEVICE, non_blocking=True))
        result['index'].append(batch_indices.numpy())
        result['tp_probability'].append(torch.sigmoid(output['tp_logit']).cpu().numpy())
        result['predicted_timeout_no_tp'].append(output['timeout_pct'].cpu().numpy() / 100)
        result['predicted_downside_no_tp'].append(output['downside_pct'].cpu().numpy() / 100)
    return {key: np.concatenate(value) for key, value in result.items()}


def conditional_utility(probability, timeout_no_tp, downside_no_tp, downside_penalty=0.0):
    return (
        probability * 0.03 + (1 - probability) * timeout_no_tp
        - downside_penalty * (1 - probability) * np.maximum(-downside_no_tp, 0)
    )


def spearman(actual, predicted):
    return float(pd.Series(actual).corr(pd.Series(predicted), method='spearman'))


def conditional_metrics(prediction):
    index = prediction['index']
    probability = prediction['tp_probability']
    negative = y[index] == 0
    utility = conditional_utility(
        probability, prediction['predicted_timeout_no_tp'], prediction['predicted_downside_no_tp'],
    )
    return {
        'samples': int(len(index)), 'positives': int(y[index].sum()), 'prevalence': float(y[index].mean()),
        'pr_auc': float(average_precision_score(y[index], probability)),
        'roc_auc': float(roc_auc_score(y[index], probability)),
        'brier': float(brier_score_loss(y[index], probability)),
        'timeout_no_tp_mae': float(mean_absolute_error(timeout_return[index][negative], prediction['predicted_timeout_no_tp'][negative])),
        'timeout_no_tp_spearman': spearman(timeout_return[index][negative], prediction['predicted_timeout_no_tp'][negative]),
        'downside_no_tp_mae': float(mean_absolute_error(future_downside[index][negative], prediction['predicted_downside_no_tp'][negative])),
        'downside_no_tp_spearman': spearman(future_downside[index][negative], prediction['predicted_downside_no_tp'][negative]),
        'utility_spearman': spearman(trade_return[index], utility),
    }


def fit_conditional_fold(x_scaled, train_indices, valid_indices, periods, fold_seed):
    torch.manual_seed(fold_seed)
    model = MultiTaskMPTSNet(x_scaled.shape[-1], x_scaled.shape[1], periods).to(DEVICE)
    train_y = y[train_indices]
    pos_weight_value = float((len(train_y) - train_y.sum()) / max(train_y.sum(), 1))
    tp_criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight_value, device=DEVICE))
    timeout_criterion = nn.HuberLoss(delta=1.0)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS, eta_min=3e-5)
    generator = torch.Generator().manual_seed(fold_seed)
    loader = DataLoader(
        MultiTaskDataset(x_scaled, train_indices), batch_size=BATCH_SIZE, shuffle=True,
        generator=generator, pin_memory=DEVICE.type == 'cuda',
    )
    best_score, best_state, best_epoch, stale = -np.inf, None, 0, 0
    history = []
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        totals = {'loss': 0.0, 'tp': 0.0, 'timeout': 0.0, 'downside': 0.0}
        for batch_x, batch_y, batch_timeout, batch_downside, _ in loader:
            batch_x = batch_x.to(DEVICE, non_blocking=True)
            batch_y = batch_y.to(DEVICE, non_blocking=True)
            batch_timeout = batch_timeout.to(DEVICE, non_blocking=True)
            batch_downside = batch_downside.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            output = model(batch_x)
            tp_loss = tp_criterion(output['tp_logit'], batch_y)
            negative = batch_y.eq(0)
            timeout_loss = timeout_criterion(output['timeout_pct'][negative], batch_timeout[negative])
            downside_loss = pinball_loss(
                output['downside_pct'][negative], batch_downside[negative], DOWNSIDE_QUANTILE,
            )
            loss = tp_loss + TIMEOUT_LOSS_WEIGHT * timeout_loss + DOWNSIDE_LOSS_WEIGHT * downside_loss
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            for key, value in [('loss', loss), ('tp', tp_loss), ('timeout', timeout_loss), ('downside', downside_loss)]:
                totals[key] += value.item() * len(batch_x)
        valid_prediction = predict_conditional(model, x_scaled, valid_indices)
        valid_metrics = conditional_metrics(valid_prediction)
        selection_score = valid_metrics['pr_auc'] + 0.25 * valid_metrics['utility_spearman']
        history.append({
            'epoch': epoch, **{f'train_{key}_loss': value / len(train_indices) for key, value in totals.items()},
            **{f'valid_{key}': value for key, value in valid_metrics.items()},
            'selection_score': selection_score, 'learning_rate': optimizer.param_groups[0]['lr'],
        })
        if selection_score > best_score + 1e-4:
            best_score = selection_score
            best_epoch = epoch
            best_state = {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}
            stale = 0
        else:
            stale += 1
        scheduler.step()
        if stale >= PATIENCE:
            break
    model.load_state_dict(best_state)
    return model, pd.DataFrame(history), pos_weight_value, best_epoch


fold_rows, histories, oof_frames = [], [], []
final_bundle = None
started = time.time()
for fold_number, valid_session in enumerate(OOF_VALID_SESSIONS, start=1):
    train_sessions = [session for session in sessions if session < valid_session]
    train_indices = np.flatnonzero(metadata['session'].isin(train_sessions).to_numpy())
    valid_indices = np.flatnonzero(metadata['session'].eq(valid_session).to_numpy())
    center, scale = fit_robust_scaler(X, train_indices)
    X_scaled = apply_scaler(X, center, scale)
    periods = discover_periods(X_scaled, train_indices)
    model, history, pos_weight, best_epoch = fit_conditional_fold(
        X_scaled, train_indices, valid_indices, periods, SEED + 200 + fold_number,
    )
    train_prediction = predict_conditional(model, X_scaled, train_indices)
    valid_prediction = predict_conditional(model, X_scaled, valid_indices)
    train_metrics = conditional_metrics(train_prediction)
    valid_metrics = conditional_metrics(valid_prediction)
    fold_rows.append({
        'fold': fold_number, 'valid_session': valid_session, 'train_sessions': train_sessions,
        'periods': periods, 'pos_weight': pos_weight, 'best_epoch': best_epoch,
        **{f'train_{key}': value for key, value in train_metrics.items()},
        **{f'valid_{key}': value for key, value in valid_metrics.items()},
    })
    history['fold'] = fold_number
    history['valid_session'] = valid_session
    histories.append(history)
    frame = metadata.iloc[valid_prediction['index']].copy()
    frame['fold'] = fold_number
    for key in ['tp_probability', 'predicted_timeout_no_tp', 'predicted_downside_no_tp']:
        frame[key] = valid_prediction[key]
    oof_frames.append(frame)
    if valid_session == sessions[-2]:
        final_bundle = {
            'state_dict': {name: value.detach().cpu().clone() for name, value in model.state_dict().items()},
            'center': center.copy(), 'scale': scale.copy(), 'periods': periods,
            'train_sessions': train_sessions, 'calibration_session': valid_session,
            'pos_weight': pos_weight, 'best_epoch': best_epoch,
        }
    print(
        f"fold {fold_number}/{len(OOF_VALID_SESSIONS)} {valid_session} | epoch={best_epoch} | "
        f"AP {train_metrics['pr_auc']:.4f}->{valid_metrics['pr_auc']:.4f} | "
        f"utility rho {train_metrics['utility_spearman']:.3f}->{valid_metrics['utility_spearman']:.3f}"
    )
    del model, X_scaled
    torch.cuda.empty_cache()
    gc.collect()

fold_metrics = pd.DataFrame(fold_rows)
training_history = pd.concat(histories, ignore_index=True)
oof = pd.concat(oof_frames).sort_values('entry_timestamp').reset_index(drop=True)
print(f'elapsed: {(time.time() - started) / 60:.2f} min')
display(fold_metrics[[
    'fold', 'valid_session', 'best_epoch', 'train_pr_auc', 'valid_pr_auc',
    'valid_timeout_no_tp_spearman', 'valid_downside_no_tp_spearman', 'valid_utility_spearman',
]])

/home/user/anaconda3/envs/urban/lib/python3.12/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


fold 1/5 session_2026-07-10 | epoch=18 | AP 0.4068->0.1111 | utility rho 0.201->0.091


/home/user/anaconda3/envs/urban/lib/python3.12/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


fold 2/5 session_2026-07-13 | epoch=6 | AP 0.1436->0.1536 | utility rho -0.021->-0.041


/home/user/anaconda3/envs/urban/lib/python3.12/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


...

/home/user/anaconda3/envs/urban/lib/python3.12/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


fold 4/5 session_2026-07-15 | epoch=16 | AP 0.3015->0.1145 | utility rho 0.170->0.044


/home/user/anaconda3/envs/urban/lib/python3.12/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


fold 5/5 session_2026-07-16 | epoch=10 | AP 0.2380->0.0889 | utility rho 0.089->0.032
elapsed: 3.61 min


,fold,valid_session,best_epoch,train_pr_auc,valid_pr_auc,valid_timeout_no_tp_spearman,valid_downside_no_tp_spearman,valid_utility_spearman
0,1,session_2026-07-10,18,0.406780,0.111110,0.280469,0.664765,0.090804
1,2,session_2026-07-13,6,0.143639,0.153626,0.264734,0.611257,-0.041045
2,3,session_2026-07-14,8,0.177209,0.120301,0.251835,0.570504,-0.002731
3,4,session_2026-07-15,16,0.301490,0.114497,0.276540,0.630536,0.044080
4,5,session_2026-07-16,10,0.237954,0.088950,0.288719,0.660169,0.031653


## OOF buy policy와 holdout Test

penalty와 상위 진입 비율은 5일 OOF에서 선택한다. 매도 모델용 데이터는 이 OOF score를 저장하고, 다음 notebook에서 날짜별 threshold를 적용한다.

In [3]:
def cooldown_trades(frame, signal):
    work = frame.copy()
    work['_signal'] = np.asarray(signal, dtype=bool)
    selected_indices = []
    for _, group in work.sort_values('entry_timestamp').groupby(['session', 'symbol'], sort=False):
        available_at = None
        for index, row in group.iterrows():
            if not row['_signal']:
                continue
            if available_at is not None and row['entry_timestamp'] < available_at:
                continue
            selected_indices.append(index)
            hold = int(row['first_hit_minute']) if row['target_net3_within_5m'] == 1 else 5
            available_at = row['entry_timestamp'] + pd.Timedelta(minutes=hold)
    trades = work.loc[selected_indices].sort_values('entry_timestamp').copy()
    trades['trade_return'] = np.where(
        trades['target_net3_within_5m'].eq(1), 0.03, trades['timeout_net_return_5m'],
    )
    trades['pnl_usd'] = STAKE_USD * trades['trade_return']
    return trades


def policy_metrics(frame, signal):
    trades = cooldown_trades(frame, signal)
    if len(trades) == 0:
        return {'trades': 0, 'sessions_traded': 0, 'precision': np.nan, 'win_rate': np.nan,
                'mean_return': np.nan, 'risk_adjusted_return': np.nan, 'total_pnl_usd': np.nan,
                'profit_factor': np.nan, 'max_drawdown_usd': np.nan}
    returns = trades['trade_return'].to_numpy()
    pnl = trades['pnl_usd'].to_numpy()
    equity = np.cumsum(pnl)
    peak = np.maximum.accumulate(np.r_[0.0, equity])
    gross_profit = pnl[pnl > 0].sum()
    gross_loss = -pnl[pnl < 0].sum()
    return {
        'trades': int(len(trades)), 'sessions_traded': int(trades['session'].nunique()),
        'precision': float(trades['target_net3_within_5m'].mean()),
        'win_rate': float((returns > 0).mean()), 'mean_return': float(returns.mean()),
        'risk_adjusted_return': float(returns.mean() - returns.std(ddof=1) / math.sqrt(len(returns))) if len(returns) > 1 else float(returns.mean()),
        'total_pnl_usd': float(pnl.sum()),
        'profit_factor': float(gross_profit / gross_loss) if gross_loss > 0 else float('inf'),
        'max_drawdown_usd': float((np.r_[0.0, equity] - peak).min()),
    }


TOP_FRACTIONS = [0.001, 0.0025, 0.005, 0.01, 0.02, 0.03, 0.05, 0.075, 0.10]
DOWNSIDE_PENALTIES = [0.0, 0.10, 0.25, 0.50, 1.0]
candidate_rows = []
for penalty in DOWNSIDE_PENALTIES:
    score = conditional_utility(
        oof['tp_probability'].to_numpy(), oof['predicted_timeout_no_tp'].to_numpy(),
        oof['predicted_downside_no_tp'].to_numpy(), penalty,
    )
    for top_fraction in TOP_FRACTIONS:
        signal = np.zeros(len(oof), dtype=bool)
        thresholds = []
        for fold, group in oof.groupby('fold'):
            fold_score = score[group.index]
            threshold = float(np.quantile(fold_score, 1 - top_fraction))
            thresholds.append(threshold)
            signal[group.index] = fold_score >= threshold
        candidate_rows.append({
            'downside_penalty': penalty, 'top_fraction': top_fraction,
            'mean_fold_threshold': float(np.mean(thresholds)), **policy_metrics(oof, signal),
        })
candidate_table = pd.DataFrame(candidate_rows)
eligible = candidate_table[
    candidate_table['trades'].ge(MIN_OOF_TRADES) & candidate_table['sessions_traded'].ge(3)
]
if eligible.empty:
    eligible = candidate_table[candidate_table['trades'].gt(0)]
selected = eligible.sort_values(['risk_adjusted_return', 'total_pnl_usd']).iloc[-1]
SELECTED_TOP_FRACTION = float(selected['top_fraction'])
SELECTED_DOWNSIDE_PENALTY = float(selected['downside_penalty'])
DEPLOYMENT_ELIGIBLE = bool(
    selected['risk_adjusted_return'] > 0 and selected['total_pnl_usd'] > 0 and selected['profit_factor'] > 1
)
oof['buy_score'] = conditional_utility(
    oof['tp_probability'].to_numpy(), oof['predicted_timeout_no_tp'].to_numpy(),
    oof['predicted_downside_no_tp'].to_numpy(), SELECTED_DOWNSIDE_PENALTY,
)
oof['buy_signal'] = False
oof['sell_training_candidate'] = False
SELL_TRAIN_TOP_FRACTION = max(SELECTED_TOP_FRACTION, 0.05)
for fold, group in oof.groupby('fold'):
    policy_threshold = group['buy_score'].quantile(1 - SELECTED_TOP_FRACTION)
    training_threshold = group['buy_score'].quantile(1 - SELL_TRAIN_TOP_FRACTION)
    oof.loc[group.index, 'buy_signal'] = group['buy_score'].ge(policy_threshold)
    oof.loc[group.index, 'sell_training_candidate'] = group['buy_score'].ge(training_threshold)
print('selected:', SELECTED_DOWNSIDE_PENALTY, SELECTED_TOP_FRACTION, 'deploy:', DEPLOYMENT_ELIGIBLE)
print('sell-training broad fraction:', SELL_TRAIN_TOP_FRACTION)
display(candidate_table.sort_values(['risk_adjusted_return', 'total_pnl_usd'], ascending=False).head(15))

calibration = oof[oof['session'].eq(final_bundle['calibration_session'])]
CALIBRATION_THRESHOLD = float(calibration['buy_score'].quantile(1 - SELECTED_TOP_FRACTION))
test_indices = np.flatnonzero(metadata['session'].eq(TEST_SESSION).to_numpy())
X_final_scaled = apply_scaler(X, final_bundle['center'], final_bundle['scale'])
final_model = MultiTaskMPTSNet(X.shape[-1], X.shape[1], final_bundle['periods']).to(DEVICE)
final_model.load_state_dict(final_bundle['state_dict'])
test_prediction = predict_conditional(final_model, X_final_scaled, test_indices)
test_predictions = metadata.iloc[test_prediction['index']].copy().reset_index(drop=True)
for key in ['tp_probability', 'predicted_timeout_no_tp', 'predicted_downside_no_tp']:
    test_predictions[key] = test_prediction[key]
test_predictions['buy_score'] = conditional_utility(
    test_predictions['tp_probability'].to_numpy(), test_predictions['predicted_timeout_no_tp'].to_numpy(),
    test_predictions['predicted_downside_no_tp'].to_numpy(), SELECTED_DOWNSIDE_PENALTY,
)
test_predictions['buy_signal'] = test_predictions['buy_score'].ge(CALIBRATION_THRESHOLD)
test_metrics = conditional_metrics(test_prediction)
test_policy = policy_metrics(test_predictions, test_predictions['buy_signal'].to_numpy())
test_trades = cooldown_trades(test_predictions, test_predictions['buy_signal'].to_numpy())

model_path = MODEL_ROOT / f'{MODEL_VERSION}.pt'
metrics_path = RESULT_ROOT / f'{MODEL_VERSION}_metrics.json'
oof_path = RESULT_ROOT / f'{MODEL_VERSION}_oof_predictions.parquet'
test_path = RESULT_ROOT / f'{MODEL_VERSION}_test_predictions.parquet'
torch.save({
    'model_version': MODEL_VERSION, 'architecture': 'ConditionalBuyMPTSNet',
    'state_dict': final_bundle['state_dict'], 'input_dim': X.shape[-1], 'seq_len': X.shape[1],
    'periods': final_bundle['periods'], 'feature_names': schema['default_feature_names'],
    'scaler_center': torch.from_numpy(final_bundle['center'].copy()),
    'scaler_scale': torch.from_numpy(final_bundle['scale'].copy()),
    'train_sessions': final_bundle['train_sessions'], 'calibration_session': final_bundle['calibration_session'],
    'test_session': TEST_SESSION, 'selected_top_fraction': SELECTED_TOP_FRACTION,
    'selected_downside_penalty': SELECTED_DOWNSIDE_PENALTY,
    'calibration_threshold': CALIBRATION_THRESHOLD, 'deployment_eligible': DEPLOYMENT_ELIGIBLE,
}, model_path)
oof.to_parquet(oof_path, index=False, compression='zstd')
test_predictions.to_parquet(test_path, index=False, compression='zstd')
fold_metrics.to_parquet(RESULT_ROOT / f'{MODEL_VERSION}_fold_metrics.parquet', index=False, compression='zstd')
training_history.to_parquet(RESULT_ROOT / f'{MODEL_VERSION}_training_history.parquet', index=False, compression='zstd')
candidate_table.to_parquet(RESULT_ROOT / f'{MODEL_VERSION}_threshold_candidates.parquet', index=False, compression='zstd')
test_trades.to_parquet(RESULT_ROOT / f'{MODEL_VERSION}_test_trades.parquet', index=False, compression='zstd')
metrics = {
    'model_version': MODEL_VERSION, 'parameters': sum(p.numel() for p in final_model.parameters()),
    'oof_sessions': OOF_VALID_SESSIONS, 'test_session': TEST_SESSION,
    'selected_top_fraction': SELECTED_TOP_FRACTION, 'selected_downside_penalty': SELECTED_DOWNSIDE_PENALTY,
    'sell_training_top_fraction': SELL_TRAIN_TOP_FRACTION, 'calibration_threshold': CALIBRATION_THRESHOLD,
    'deployment_eligible': DEPLOYMENT_ELIGIBLE,
    'selected_oof_policy': {key: (None if pd.isna(value) else float(value)) for key, value in selected.to_dict().items()},
    'test_metrics': test_metrics, 'test_policy': test_policy,
}
metrics_path.write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding='utf-8')
print('model:', model_path)
display(pd.DataFrame([selected.to_dict(), {'top_fraction': SELECTED_TOP_FRACTION, **test_policy}], index=['OOF', 'Test']).T)

selected: 1.0 0.1 deploy: False
sell-training broad fraction: 0.1


,downside_penalty,top_fraction,mean_fold_threshold,trades,sessions_traded,precision,win_rate,mean_return,risk_adjusted_return,total_pnl_usd,profit_factor,max_drawdown_usd
18,0.25,0.001,0.024655,28,5,0.142857,0.428571,-0.005744,-0.011541,-16.083542,0.632984,-18.337332
0,0.00,0.001,0.026067,27,5,0.111111,0.370370,-0.006944,-0.012352,-18.748928,0.552700,-19.363127
9,0.10,0.001,0.025496,28,5,0.142857,0.392857,-0.006974,-0.012732,-19.527137,0.572654,-20.141336
44,1.00,0.100,0.006481,1326,5,0.121418,0.380090,-0.013139,-0.014352,-1742.202268,0.409912,-1745.202268
42,1.00,0.050,0.011264,765,5,0.125490,0.402614,-0.012909,-0.014528,-987.529293,0.427823,-987.882509
35,0.50,0.100,0.012844,1313,5,0.126428,0.377761,-0.013394,-0.014608,-1758.679511,0.406122,-1761.679511
33,0.50,0.050,0.016489,763,5,0.128440,0.401048,-0.013082,-0.014734,-998.147455,0.433684,-1004.379208
29,0.50,0.005,0.021709,117,5,0.128205,0.401709,-0.010922,-0.014757,-127.790691,0.494127,-135.895669
43,1.00,0.075,0.008801,1072,5,0.122201,0.385261,-0.013382,-0.014768,-1434.499216,0.415411,-1437.499216
24,0.25,0.050,0.019146,761,5,0.124836,0.399474,-0.013201,-0.014861,-1004.583438,0.432087,-1008.685851


/home/user/anaconda3/envs/urban/lib/python3.12/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


model: /home/user/urbandatalab/YSLee/model/conditional_buy_mptsnet_v1.pt


,OOF,Test
downside_penalty,1.000000,NaN
top_fraction,0.100000,0.100000
mean_fold_threshold,0.006481,NaN
trades,1326.000000,180.000000
sessions_traded,5.000000,1.000000
precision,0.121418,0.022222
win_rate,0.380090,0.311111
mean_return,-0.013139,-0.016172
risk_adjusted_return,-0.014352,-0.018934
total_pnl_usd,-1742.202268,-291.093123
